# 任务
使用“meta-llama/Llama-3.2-3B-Instruct”模型为业务用例（例如测试 HR 系统）创建合成数据生成器。

首先，使用“BitsAndBytesConfig”4 位量化 (NF4) 初始化模型，以实现高效执行。然后，实现一个函数来生成结构化综合记录（例如员工数据）并将其解析为 Pandas DataFrame。最后，构建一个 Gradio 界面，允许用户选择数据类别和记录数，并将结果显示在表格中以供导出。

## 定义数据策略

### 子任务：
概述合成数据生成器的数据架构、格式和业务目的。

### 综合数据策略：HR 系统测试

#### 1. 商业目的
主要目标是生成用于测试人力资源管理系统 (HRIS) 的高质量综合 HR 数据。这样可以进行性能测试、UI 开发和分析原型设计，而无需暴露敏感的个人身份信息 (PII)。

#### 2. 数据架构（员工记录）
每个生成的记录将代表一个具有以下字段的“员工”：
- **员工 ID**：唯一标识符（例如 EMP-001）。
- **全名**：随机生成的真实姓名。
- **部门**：以下之一：工程、销售、营销、人力资源、财务或法律。
- **职位**：与角色相适应的职位（例如，软件工程师、客户经理）。
- **薪资**：实际范围内的数值（$40,000 - $200,000）。
- **雇用日期**：日期范围从 2010 年至今。
- **性能评级**：分类值（1-5 或优秀、良好、一般等）。

#### 3. 输出格式
为了确保与下游分析的兼容性，模型必须以 **JSON** 或 **CSV** 格式输出数据。这允许无缝加载到“pandas.DataFrame”中以进行进一步处理。

#### 4. 约束和分布
- **实际薪资**：薪资应与职位和资历大致相关。
- **部门平衡**：记录应跨部门分发，以避免偏差，除非另有说明。
- **唯一性**：员工 ID 在数据集中必须是唯一的。

## 设置量化模型

使用 BitsAndBytes 4 位量化 (NF4) 初始化 Llama-3.2-3B-Instruct 模型，以便在 Colab 中高效执行。

In [ ]:
!uv install -U bitsandbytes>=0.46.1

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
from google.colab import userdata
from huggingface_hub import login

# 1. 定义模型ID
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

# 2. 配置 BitsAndBytes 进行 4 位 NF4 量化
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 3. 获取Token并登录
hf_token = userdata.get('HF_TOKEN')
if hf_token:
    login(token=hf_token, add_to_git_credential=True)

# 4. 加载分词器
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token

# 5. 负载量化模型
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    token=hf_token
)

print(f"Model {MODEL_ID} and tokenizer loaded successfully with 4-bit NF4 quantization.")

## 实现生成器逻辑

创建一个函数来使用量化的 Llama 模型生成合成记录，并将结果解析为 Pandas DataFrame。

In [2]:
import json
import re
import pandas as pd

def generate_synthetic_data(category, num_records):
    """
    Generates synthetic records using the Llama model and parses them into a DataFrame.
    """
    # 1. 根据HR策略构建提示
    prompt = f"""Generate a valid JSON list containing {num_records} synthetic {category} records.
Each record must strictly follow this schema:
- Employee ID: Unique string (e.g., EMP-001)
- Full Name: Realistic name
- Department: Engineering, Sales, Marketing, HR, Finance, or Legal
- Job Title: Appropriate for the department
- Salary: Integer between 40000 and 200000
- Hire Date: YYYY-MM-DD between 2010 and 2024
- Performance Rating: Integer 1-5

Return ONLY the JSON list. Do not include any explanations or markdown code blocks."""

    # 2. Token化并生成
    messages = [{"role": "user", "content": prompt}]
    # 从 apply_chat_template 返回的 BatchEncoding 对象中提取“input_ids”张量
    input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt")['input_ids'].to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            max_new_tokens=1000,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

    # 3. 解码响应
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # 4.使用正则表达式提取JSON内容（查找[]之间的内容）
    try:
        json_match = re.search(r'\[.*\d.*\]', response, re.DOTALL)
        if json_match:
            json_str = json_match.group(0)
            data = json.loads(json_str)
        else:
            # 如果正则表达式失败但响应是原始 JSON，则回退到直接加载
            data = json.loads(response)

        # 5. 转换为DataFrame
        df = pd.DataFrame(data)
        return df
    except Exception as e:
        print(f"Error parsing model output: {e}")
        print("Raw Response:", response)
        return pd.DataFrame()

# 用 3 条记录测试该功能
test_df = generate_synthetic_data('HR/Employee', 3)
print("Generated Synthetic Data:")
display(test_df)

将其保存到 Google 表格中应该很容易

In [1]:
# 下方为可执行代码（逻辑与字符串保持原文，便于运行）
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=test_df)